# case02 匿名加工情報（ID-POS）— 加工処理デモ

**所要時間: 約10分** ／ ゴール: 購買データ（ID-POS）を **匿名加工情報**に加工し、
「元に戻せない・特異な個人が浮かない」状態にしたうえで、**属性 × 購買傾向**の分析が
成立することを確認する。

**流れ（5ステップ）**
1. 仮IDへ置換（連結符号を断つ / 施行規則34条3号）
2. 直接識別子の削除（氏名・電話番号 / 34条1号）
3. 準識別子の丸め（年代・市区郡・分単位 / 5号措置）
4. 特異な記述の処理（商品カテゴリ化・数量上限・金額区分 / 34条4号）
5. 該当者の人数を確認 → 属性 × 購買傾向

> 設計の正本はサイトの [① 検討](https://gghatano.github.io/pets-seminar-01/case02_anonymized/05_processing_design/) を参照。データはすべて教材用の合成データ。

In [ ]:
import numpy as np
import pandas as pd

# 生成済みダミーデータの場所（GitHub raw）。ローカル実行時はこの URL を差し替える。
DATA_BASE = (
    "https://raw.githubusercontent.com/gghatano/pets-seminar-01/main/data/case02_anonymized/raw"
)

customers = pd.read_csv(f"{DATA_BASE}/customers.csv")
transactions = pd.read_csv(f"{DATA_BASE}/transactions.csv")
purchases = pd.read_csv(f"{DATA_BASE}/purchases.csv")

REFERENCE_YEAR = 2026  # 年齢計算の基準年（生成時の REFERENCE_DATE に対応）
print(len(customers), len(transactions), len(purchases))
customers.head(3)

## 加工前の確認 — 特異値をのぞいてみる
施行規則34条4号「特異な記述等」の対象になりそうな外れ値が含まれている。

In [ ]:
# 超高齢者（生存者が極めて少ない生年月日）
ages = REFERENCE_YEAR - customers["生年月日"].str[:4].astype(int)
print("最高齢:", ages.max(), "歳 /", (ages >= 100).sum(), "名が100歳以上")

# 希少・超高額な購買、大量購入
display(purchases[purchases["金額"] >= 10000])
display(purchases[purchases["数量"] >= 40])

## STEP 1: 会員ID → 仮ID（34条3号：連結符号の削除）

会員IDは、元データと加工後データを**つなぎ直せる符号**。ランダムに割り当てた**仮ID**へ
置き換え、**元IDと仮IDの対応表は保存しない**（保存すれば連結符号になってしまう）。
加工後は仮IDで顧客と購買を結合できるが、元の個人へはたどれない。

In [ ]:
rng = np.random.default_rng(20260719)
members = customers["会員ID"].tolist()
order = rng.permutation(len(members))  # 会員ID順と別の順序で仮IDを割当て
id_map = {members[o]: f"A{i + 1:06d}" for i, o in enumerate(order)}

customers["仮ID"] = customers["会員ID"].map(id_map)
transactions["仮ID"] = transactions["会員ID"].map(id_map)
customers[["会員ID", "仮ID"]].head(3)

## STEP 2: 直接識別子の削除（34条1号）
氏名・電話番号は単体で個人を特定できる／本人に到達できるため削除する。

In [ ]:
# 年代・市区郡は次のSTEPで作るので、ここでは削除対象だけ落とす
customers_anon = customers.drop(columns=["会員ID", "氏名", "電話番号"])
customers_anon.columns.tolist()

## STEP 3: 準識別子の丸め（5号措置：該当者の人数に応じた一般化）

生年月日・住所・利用日時を細かく残すと「その組合せに1人」の人が再識別される。
**同じ属性の人が十分いる**粒度まで丸める。

In [ ]:
# 生年月日 → 年代7区分
age = REFERENCE_YEAR - customers["生年月日"].str[:4].astype(int)
bins = [0, 20, 30, 40, 50, 60, 70, 200]
labels = ["20歳未満", "20代", "30代", "40代", "50代", "60代", "70歳以上"]
customers_anon["年代"] = pd.cut(age, bins=bins, right=False, labels=labels)

# 住所 → 市区郡（都道府県 + 市区郡まで。政令市は「◯◯市◯◯区」まで）
pat = r"^(\D+?[都道府県](?:\D+?市\D+?区|\D+?[市区郡]))"
customers_anon["市区郡"] = customers["住所"].str.extract(pat)
customers_anon = customers_anon.drop(columns=["生年月日", "住所"])
customers_anon = customers_anon[["仮ID", "年代", "性別", "市区郡"]]

# 利用日時 → 分単位（秒を削除）
transactions["利用日時"] = pd.to_datetime(transactions["利用日時"]).dt.strftime("%Y-%m-%d %H:%M")
customers_anon.head(3)

## STEP 4: 特異な記述の処理（34条4号）＋ 不要IDの削除

- **商品名 → 商品カテゴリ**（限定品・超高級品もカテゴリに吸収され、特異性が消える）
- **数量**は上限で丸め（大量購入の特異性を除去）
- **金額 → 金額区分**（超高額はトップコーディングで「5,000円以上」に吸収）
- 提供先に不要なID（店舗ID・取引ID・担当者ID・商品ID・明細ID）は削除

In [ ]:
# 商品ID → カテゴリの対応（希少品 G90-92 もカテゴリへ）
CATEGORY = {
    "G01": "野菜",
    "G02": "野菜",
    "G03": "果物",
    "G04": "果物",
    "G05": "精肉",
    "G06": "精肉",
    "G07": "鮮魚",
    "G08": "鮮魚",
    "G09": "米・穀物",
    "G10": "惣菜・パン",
    "G11": "惣菜・パン",
    "G12": "飲料",
    "G13": "飲料",
    "G14": "菓子",
    "G15": "菓子",
    "G16": "調味料",
    "G90": "果物",
    "G91": "菓子",
    "G92": "鮮魚",
}

# 購買履歴に取引側の情報（仮ID・利用日時・店舗名）を結合
pu = purchases.merge(
    transactions[["取引ID", "仮ID", "利用日時", "店舗名"]], on="取引ID", how="left"
)
pu["商品カテゴリ"] = pu["商品ID"].map(CATEGORY)
pu["数量"] = pu["数量"].clip(upper=5)  # トップコーディング（大量購入の特異性を除去）
amt_bins = [0, 500, 1000, 2000, 5000, np.inf]
amt_labels = ["〜499", "500〜999", "1,000〜1,999", "2,000〜4,999", "5,000円以上"]
pu["金額区分"] = pd.cut(pu["金額"], bins=amt_bins, right=False, labels=amt_labels)

purchases_anon = pu[["仮ID", "利用日時", "店舗名", "商品カテゴリ", "数量", "金額区分"]]
purchases_anon.head(3)

## 加工前後の比較
顧客属性は 6項目 → 4項目に。直接識別子・連結符号・特異値が消え、分析に使う粒度だけが残る。

In [ ]:
print(
    "加工前 customers:",
    list(customers[["会員ID", "氏名", "生年月日", "性別", "電話番号", "住所"]].columns),
)
print("加工後 customers_anon:", list(customers_anon.columns))
print("加工後 purchases_anon:", list(purchases_anon.columns))
purchases_anon.head(5)

## 確認テスト（匿名加工が要件を満たすか）

In [ ]:
# 直接識別子・連結符号（元会員ID）・生の準識別子が残っていない
for col in ["会員ID", "氏名", "電話番号", "生年月日", "住所"]:
    assert col not in customers_anon.columns, col
assert "仮ID" in customers_anon.columns
assert customers_anon["仮ID"].is_unique and len(customers_anon) == 900

# 利用日時に秒が残っていない（分単位）
assert purchases_anon["利用日時"].str.match(r"^\d{4}-\d\d-\d\d \d\d:\d\d$").all()

# 特異値の処理: 商品名は消えてカテゴリ化、数量は上限5、金額は区分
assert "商品名" not in purchases_anon.columns and "商品カテゴリ" in purchases_anon.columns
assert purchases_anon["数量"].max() <= 5
assert "金額" not in purchases_anon.columns and "金額区分" in purchases_anon.columns

# 件数は維持されている
assert len(purchases_anon) == len(purchases)
print("すべての確認テストに合格 ✓")

## 該当者の人数を確認する — 丸めの度合が十分か

準識別子の組合せ **(年代 × 性別 × 市区郡)** ごとに人数を数える。レポートは、丸めの度合を
**該当者の人数・データセットの大きさ・人口の多寡に応じて（可変に）判断**するとしている。
該当者が少ない組合せが出ていないかを見る。

In [ ]:
grp = customers_anon.groupby(["年代", "性別", "市区郡"], observed=True).size()
few = 5  # 「該当者が少ない」の目安（固定の基準値ではない）
print("組合せの数:", len(grp))
print("最も少ない組合せの人数:", int(grp.min()))
print(
    f"該当者が {few} 名未満の組合せ:",
    int((grp < few).sum()),
    "組合せ /",
    int(grp[grp < few].sum()),
    "名",
)
print("→ 本デモは900件と小規模なため小さな組合せが出やすい。")
print(
    "   実務では、丸めを粗くする（市区郡→都道府県 等）／データを増やす／該当値を削除 で調整する。"
)
grp.sort_values().head(5)

## 加工後データによる分析（属性 × 購買傾向）

匿名加工の後でも、提供先が求める **「どんな属性の人が、どんな商品を、どのくらい買うか」**
の分析は成立する。仮IDで顧客属性と購買履歴を結合して確認する。

In [ ]:
m = purchases_anon.merge(customers_anon, on="仮ID", how="left")

print("■ 年代別 購入カテゴリ トップ3")
for age_band in ["20代", "40代", "70歳以上"]:
    top = m[m["年代"] == age_band]["商品カテゴリ"].value_counts().head(3).index.tolist()
    print(f"  {age_band}: {top}")

print("\n■ 年代別 金額区分の構成（割合）")
tab = pd.crosstab(m["年代"], m["金額区分"], normalize="index").round(2)
display(tab)

## まとめ

- **仮IDへの置換 + 対応表を残さない**ことで、元の個人へはたどれない（34条3号）。
- **直接識別子の削除**（34条1号）、**準識別子の丸め**（5号措置）、**特異な記述の処理**（34条4号）で、
  「1人しかいない組合せ」から個人が浮かないようにした。
- それでも **属性 × 購買傾向** の分析は成立する＝匿名加工情報として第三者提供に使える形。
- 実際の提供では、**含まれる情報の項目の公表**や**識別行為の禁止**などの取り扱いも必要（詳細は最新のガイドライン）。

結果ページ: [③ 結果](https://gghatano.github.io/pets-seminar-01/case02_anonymized/09_results/)